<a href="https://colab.research.google.com/github/ghadialbishri/AIEP_Olo_student/blob/master/week_6_llms/labs/D1_lm_perplexity/D1_lm_perplexity_blank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W6D1 — Language Modeling: Tokens, Perplexity, Decoding — Lab

**Week 6 · Day 1 · LLMs** · Lab

> **The big idea.** A language model is a function that reads text, chops it into
> **tokens**, and outputs a probability for every possible next token. Everything
> else — perplexity, text generation, hallucinations, long contexts, Arabic feeling
> "clunky" — is a consequence of those three steps.

Last week on **W5** you got comfortable with word embeddings and classical NLP. Today
you meet the modern descendant: a **pretrained Transformer language model**. You will
not train one — you will **probe one**. By the end you will have measured, with your
own hands, exactly how a GPT-2 scale model sees text, how confident it is on
different kinds of sentences, which specific token in a sentence surprised it most,
and how much the decoding strategy changes what it "says".

<div dir="rtl" align="right">

> **الفكرة الكبرى.** النموذج اللغويّ هو دالّة تقرأ نصًّا، وتقسّمه إلى **رموز
> (tokens)**، وتُخرج احتماليّة لكلّ رمز تالٍ ممكن. وكل ما سواه — الحيرة (perplexity)،
> التوليد، الهلوسات، النوافذ الطويلة، وحتى شعور العربيّة بأنّها "ثقيلة" — نتيجة
> لهذه الخطوات الثلاث.

في **الأسبوع الخامس** تعرّفت على التضمينات الكلاسيكيّة. اليوم تلتقي بسليلها الحديث:
**نموذج Transformer لغويّ مُدرَّب مسبقًا**. لن تدرّب نموذجًا — بل ستفحصه. في نهاية
الدرس ستكون قد قست بيديك كيف يرى نموذج بحجم GPT-2 النصّ، وكم يكون واثقًا من جمل
مختلفة، وأيّ رمز بالضبط فاجأه أكثر، وكم تتغيّر مخرجاته باختلاف **استراتيجيّة فكّ
الترميز**.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

Concretely, you can:

- **Tokenize** the same meaning in Arabic and English and explain *why* the Arabic
  version almost always uses more tokens.
- Compute the **negative log-likelihood** of a sentence, convert it to **perplexity**,
  and read PPL as a measure of model surprise.
- Rank several sentences (natural, scrambled, out-of-domain) by their PPL and
  predict the ranking before running the code.
- Identify the **single most surprising token** in a sentence by inspecting per-token loss.
- Measure the **per-token cost** of running an English LLM on Arabic text (both in
  token count *and* in perplexity).
- Generate text with **greedy, temperature, top-k, and top-p** decoding and describe
  one failure mode of each.
- Tune decoding settings for two **different** use-cases (factual vs creative) and
  explain why there is no single "best" setting.

<div dir="rtl" align="right">

## الأهداف التعلّمية

بالتحديد، تستطيع:

- **تقطيع** المعنى نفسه بالعربيّة والإنجليزيّة وشرح *لماذا* تأخذ العربيّة رموزًا أكثر.
- حساب **اللوغاريتم السالب للاحتمال** لجملة، وتحويله إلى **حيرة (perplexity)**.
- ترتيب عدّة جمل (طبيعيّة، مخلوطة، خارج النطاق) حسب PPL والتنبّؤ بالترتيب قبل التشغيل.
- تحديد **الرمز الأكثر مفاجأةً** في جملة بفحص الخسارة لكل رمز.
- قياس **التكلفة لكل رمز** لتشغيل LLM إنجليزيّ على نصٍّ عربيّ (عدد الرموز + الحيرة).
- توليد نصّ بـ **greedy، temperature، top-k، top-p** ووصف وضع فشل لكلّ منها.
- ضبط إعدادات فكّ الترميز لحالتين مختلفتين (معلوماتيّة مقابل إبداعيّة) وشرح لماذا لا
  يوجد إعداد "أفضل" وحيد.

</div>


## About the model and data

**Model:** `gpt2` — the 124M-parameter small variant of OpenAI's GPT-2 (2019).
Decoder-only Transformer, autoregressive, trained on **40 GB of English web
text**. Small on purpose: it fits in under 500 MB of RAM, runs comfortably on
Colab CPU, and its weaknesses — including bad Arabic tokenization — make the
lessons visible. We also load `aubmindlab/aragpt2-base` (134M params, pretrained
on Arabic Wikipedia + OSCAR) to contrast its tokenizer and perplexity on the same
Arabic sentences.

**Data:** no download. The entire lab is driven by **small inline sentence
lists** you can read in the cells themselves:

- **3 English sentences** of varying topic (everyday, hobby, abstract).
- **3 Arabic sentences** carrying the exact same meanings — for a controlled
  cross-language comparison.
- **4 benchmark sentences** for the PPL ranking task: two natural English, one
  scrambled, one word-salad.
- **1 trick sentence** with a known-false fact ("Saudi Arabia's capital is
  Jupiter") for the per-token surprise task.

That is it. The whole point of today is to probe **the model**, not any dataset
— so the inputs are tiny, hand-picked, and chosen so you can predict the
answer before running the code.

<div dir="rtl" align="right">

## عن النموذج والبيانات

**النموذج:** `gpt2` — النسخة الصغرى من GPT-2 بـ١٢٤ مليون معامل. Transformer من
نوع decoder-only، تنبّؤي-تتابعي، مُدرَّب على **٤٠ جيجابايت من نصوص إنجليزيّة من
الويب**. صغير عمدًا: يسع في أقل من ٥٠٠ ميجابايت، يعمل على Colab CPU براحة،
وضعفه مع العربيّة يجعل الدروس ظاهرة. نحمِّل كذلك `aubmindlab/aragpt2-base`
(١٣٤M معامل، مُدرَّب على ويكيبيديا العربيّة + OSCAR) للمقارنة.

**البيانات:** بلا تنزيل. التجربة كلها مدفوعة بـ**قوائم جمل صغيرة ضمن الخلايا**:

- **٣ جمل إنجليزيّة** بمواضيع متنوّعة.
- **٣ جمل عربيّة** تحمل نفس المعاني بالضبط — لمقارنة متحكَّم بها بين اللغتين.
- **٤ جمل مرجعيّة** لمهمّة ترتيب PPL: اثنتان طبيعيّتان، واحدة مخلوطة، وواحدة
  "سلطة مفردات".
- **جملة خدعة** بحقيقة خاطئة معلومة ("عاصمة السعوديّة المشتري") لمهمّة مفاجأة
  الرموز.

هذا كلّ شيء. هدف اليوم فحص **النموذج**، لا أي بيانات — فالمدخلات صغيرة ومنتقاة
بحيث يمكنك التنبّؤ بالإجابة قبل التشغيل.

</div>


## Setup

Run this first. On Colab it `pip install`s what we need; locally it assumes the
AIEP environment is active.

<div dir="rtl" align="right">

## الإعداد

شغّل هذه الخلية أولًا. على Colab ستثبّت ما يلزم، ومحلّيًّا تفترض أن بيئة AIEP مفعّلة.

</div>


In [1]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.40", "accelerate"])

import torch, random, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__}  device={DEVICE}")

torch=2.11.0+cu130  device=cuda


In [2]:
# === Load GPT-2 small ================================================================
MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print(f"Loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"Context window:  {model.config.n_positions} tokens")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2: 124.4M params
Vocabulary size: 50,257 tokens
Context window:  1024 tokens


## Section 1 — Warm-up: three primitives  *(~10 min)*

Everything in this section already works — nothing to implement. It is a quick tour
of the three primitives you'll build on in the rest of the lab:

1. **Tokenization** — how a sentence becomes a sequence of integer tokens.
2. **Cross-entropy loss** — how surprised the model is by a sequence.
3. **Generation** — how a sequence of tokens becomes new text.

Run each cell, look at the output, and get comfortable with the three helper
functions. Section 2 is where you start measuring.

<div dir="rtl" align="right">

## القسم ١ — إحماء: الأساسيّات الثلاث  *(≈١٠ دقائق)*

كل ما في هذا القسم يعمل أصلًا — لا شيء تنفّذه. جولة سريعة في الأدوات الثلاث التي
ستبني عليها بقيّة الدرس:

١. **التقطيع** — كيف تصبح الجملة سلسلةً من الرموز الصحيحة.
٢. **خسارة الـcross-entropy** — مقدار مفاجأة النموذج بالتسلسل.
٣. **التوليد** — كيف تصبح سلسلةُ الرموز نصًّا جديدًا.

شغّل كل خلية، انظر إلى المخرجات، وتعرَّف على الدوال المساعدة الثلاث. القسم ٢ هو
حيث تبدأ القياس.

</div>


In [3]:
# === Primitive 1: tokenization ======================================================
def show_tokens(text, tok, max_show=20):
    ids = tok.encode(text)
    pieces = tok.convert_ids_to_tokens(ids)
    print(f"[{len(ids):>3} tokens]  {text[:60]!r}")
    print("   ", " | ".join(repr(p) for p in pieces[:max_show]),
          "..." if len(pieces) > max_show else "")
    return len(ids)

_ = show_tokens("The cat sat on the mat.", tokenizer)
_ = show_tokens("Transformers are language models.", tokenizer)

[  7 tokens]  'The cat sat on the mat.'
    'The' | 'Ġcat' | 'Ġsat' | 'Ġon' | 'Ġthe' | 'Ġmat' | '.' 
[  6 tokens]  'Transformers are language models.'
    'Transform' | 'ers' | 'Ġare' | 'Ġlanguage' | 'Ġmodels' | '.' 


In [4]:
# === Primitive 2: cross-entropy loss ================================================
@torch.no_grad()
def sentence_loss(text, model, tok):
    """Return the mean cross-entropy loss (nats/token) for `text` under `model`."""
    enc = tok(text, return_tensors="pt").to(model.device)
    out = model(**enc, labels=enc["input_ids"])
    return out.loss.item()

for s in ["The capital of France is Paris.",
          "The capital of France is Jupiter."]:
    print(f"loss = {sentence_loss(s, model, tokenizer):.3f} nats/token   {s!r}")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


loss = 3.798 nats/token   'The capital of France is Paris.'
loss = 5.303 nats/token   'The capital of France is Jupiter.'


In [5]:
# === Primitive 3: text generation ====================================================
@torch.no_grad()
def generate(prompt, max_new_tokens=40, model=model, tok=tokenizer, **kwargs):
    enc = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        pad_token_id=tok.eos_token_id,
        **kwargs,
    )
    text = tok.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

print(generate("Once upon a time", max_new_tokens=20, do_sample=False))

, the world was a place of great beauty and great danger. The world was a place of great


Three helpers, three primitives. Keep this picture in mind — the rest of the lab
just uses these three functions over and over, with sharper and sharper questions.

<div dir="rtl" align="right">

ثلاث دوال، ثلاث أساسيّات. احتفظ بهذه الصورة — بقيّة الدرس ليست إلا استخدام هذه
الدوال الثلاث مرّة بعد مرّة، بأسئلة أكثر حدّة.

</div>


## Section 2 — Measuring the model  *(~55 min)*

Now the hands-on part. Five tasks in order, each sharper than the last:

1. **Token counts** — same meaning, English vs Arabic. How much more does Arabic
   cost on an English-trained tokenizer?
2. **Implement `perplexity(text)`** — one line of code that turns the loss helper
   into a surprise score.
3. **Rank four sentences by PPL** — predict first, then compute.
4. **Find the most surprising token** — look inside the loss and flag the word that
   broke the model.
5. **GPT-2 vs aragpt2 on Arabic** — measure both the per-token *and* the
   per-sentence cost of using the wrong-language model.

<div dir="rtl" align="right">

## القسم ٢ — قياس النموذج  *(≈٥٥ دقيقة)*

الآن الجزء العمليّ. خمس مهام بالترتيب، كلّ واحدة أدقّ من سابقتها:

١. **عدد الرموز** — نفس المعنى، إنجليزيّ مقابل عربيّ. كم تكلف العربيّة على مُقطِّع
   مُدرَّب على الإنجليزيّة؟
٢. **تنفيذ `perplexity(text)`** — سطر واحد يحوّل الخسارة إلى درجة مفاجأة.
٣. **ترتيب ٤ جمل حسب PPL** — تنبّأ أولًا ثم احسب.
٤. **إيجاد الرمز الأكثر مفاجأةً** — انظر داخل الخسارة وأبرز الكلمة التي كسرت النموذج.
٥. **GPT-2 مقابل aragpt2 على العربيّة** — قس التكلفة لكل رمز *ولكل جملة* عند استخدام
   النموذج بلغة خاطئة.

</div>


### Task 1 — Token counts: English vs Arabic

Three short sentences carry the **same meaning** in both languages. Count tokens with
GPT-2's tokenizer, then load `aubmindlab/aragpt2-base`'s tokenizer and watch the
Arabic counts drop. Compute the two ratios.

<div dir="rtl" align="right">

### المهمّة ١ — عدد الرموز: الإنجليزيّة مقابل العربيّة

ثلاث جمل قصيرة تحمل **نفس المعنى** في كلتا اللغتين. عدّ الرموز بمُقطِّع GPT-2، ثم
حمّل مُقطِّع `aubmindlab/aragpt2-base` وشاهد الأعداد العربيّة تنخفض. احسب النسبتين.

</div>


In [6]:
# ────────────────────────────────────────────────────────────────────
# 1) Call show_tokens(s, tokenizer) in a list comprehension to collect counts.
# 2) For the Arabic-pretrained tokenizer: AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base").
# 3) Compute two ratios: (AR via GPT-2) / (EN) and (AR via GPT-2) / (AR via aragpt2).
# 4) Expected: AR / EN ratio around 2-4x; GPT-2 / aragpt2 ratio around 2-3x on Arabic.
#
# ١) نادِ show_tokens(s, tokenizer) داخل list comprehension لجمع الأعداد.
# ٢) للمُقطِّع العربيّ: AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base").
# ٣) احسب نسبتين: (AR بـGPT-2) / (EN) و (AR بـGPT-2) / (AR بـaragpt2).
# ٤) المتوقّع: AR / EN ≈ ٢–٤×؛ GPT-2 / aragpt2 ≈ ٢–٣× على العربيّة.
# ────────────────────────────────────────────────────────────────────
# ===== BEGIN MY SOLUTION =====
from transformers import AutoTokenizer

ar_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base")

SENTENCES_EN = [
    "The cat sat on the mat.",
    "I love reading books in the evening.",
    "Artificial intelligence is changing education.",
]

SENTENCES_AR = [
    "جلس القط على السجادة.",
    "أحب قراءة الكتب في المساء.",
    "الذكاء الاصطناعي يغيّر التعليم.",
]

en_counts = [len(tokenizer.encode(s)) for s in SENTENCES_EN]
ar_gpt2_counts = [len(tokenizer.encode(s)) for s in SENTENCES_AR]
ar_aragpt2_counts = [len(ar_tokenizer.encode(s)) for s in SENTENCES_AR]

print("English tokens:", en_counts)
print("Arabic tokens with GPT-2:", ar_gpt2_counts)
print("Arabic tokens with AraGPT2:", ar_aragpt2_counts)

print("Arabic/English ratio:",
      sum(ar_gpt2_counts) / sum(en_counts))

print("GPT-2/AraGPT2 Arabic ratio:",
      sum(ar_gpt2_counts) / sum(ar_aragpt2_counts))
# ===== END MY SOLUTION =====

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

English tokens: [7, 8, 7]
Arabic tokens with GPT-2: [22, 25, 31]
Arabic tokens with AraGPT2: [5, 7, 12]
Arabic/English ratio: 3.5454545454545454
GPT-2/AraGPT2 Arabic ratio: 3.25


Picking a model trained for your language is not a style choice, it is a **cost
choice**. Every API that charges per token will bill you several times more for
Arabic text sent to an English-tokenizer model, for the identical meaning.

<div dir="rtl" align="right">

اختيار نموذج مُدرَّب على لغتك ليس اختيار ذوق، بل **تكلفة**. كل واجهة تحاسب بالرمز
ستُحاسبك أضعافًا على النصّ العربيّ المُرسَل لنموذج بمُقطِّع إنجليزيّ لنفس المعنى.

</div>


### Task 2 — Implement `perplexity(text)`

**Perplexity (PPL)** is $\exp(\mathcal{L})$ where $\mathcal{L}$ is the mean per-token
cross-entropy loss. The `sentence_loss` helper already returns that mean, so this is
one line.

$$\mathrm{PPL}(x) = \exp\!\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})\right)$$

If PPL is 10, the model is "as confused as if it had to choose uniformly among 10
options" at each position. GPT-2 on well-formed English scores around 20–40. On
gibberish it rockets into the hundreds or thousands.

<div dir="rtl" align="right">

### المهمّة ٢ — نفّذ `perplexity(text)`

**الحيرة (PPL)** هي $\exp(\mathcal{L})$ حيث $\mathcal{L}$ هي متوسط خسارة
الـcross-entropy لكل رمز. `sentence_loss` تُرجع المتوسّط أصلًا، فهذا سطر واحد.

$$\mathrm{PPL}(x) = \exp\!\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t})\right)$$

إذا كانت PPL = ١٠، فالنموذج "محتار بقدر الاختيار المنتظم بين ١٠ خيارات". GPT-2 على
إنجليزيّة سليمة ≈ ٢٠–٤٠، وعلى هراء بالمئات.

</div>


In [8]:
# ────────────────────────────────────────────────────────────────────
# 1) PPL = exp(loss). One line using math.exp.
# 2) The loss returned by sentence_loss is ALREADY the per-token average.
# 3) Expected: PPL(Paris) around 20-60.
# Reference: https://huggingface.co/docs/transformers/perplexity
#
# ١) PPL = exp(loss). سطر واحد باستخدام math.exp.
# ٢) الخسارة المرجَعة من sentence_loss متوسّطة لكل رمز أصلًا.
# ٣) المتوقّع: PPL(Paris) ≈ ٢٠–٦٠.
# المرجع: https://huggingface.co/docs/transformers/perplexity
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
import math

def perplexity(text):
    return math.exp(sentence_loss(text, model, tokenizer))

_demo = "The capital of France is Paris."
print(f"perplexity('{_demo}') = {perplexity(_demo):.1f}")
# ===== END MY SOLUTION =====

perplexity('The capital of France is Paris.') = 44.6


### Task 3 — Rank four sentences by PPL

Before you run anything, **write down your predicted ranking** from lowest PPL
(model is at home) to highest PPL (model is lost). Then score, sort, and compare to
your prediction.

<div dir="rtl" align="right">

### المهمّة ٣ — رتّب أربع جمل حسب PPL

قبل التشغيل، **اكتب ترتيبك المتوقّع** من أدنى PPL إلى أعلى. ثم احسب، رتّب، وقارن
بتنبّؤك.

</div>


In [9]:
# ────────────────────────────────────────────────────────────────────
# 1) Before running, write down your PREDICTED ranking from lowest to highest PPL.
# 2) Call perplexity() on each sentence and store (sentence, ppl) pairs.
# 3) Sort ascending by PPL and print. Compare to your prediction.
#
# ١) قبل التشغيل، اكتب ترتيبك المتوقّع من الأدنى إلى الأعلى في PPL.
# ٢) نادِ perplexity() على كل جملة واحفظ أزواج (الجملة، ppl).
# ٣) رتّب تصاعديًّا حسب PPL واطبع. قارن بتنبّؤك.
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
TEST_SENTENCES = [
    "The quick brown fox jumps over the lazy dog.",
    "She opened the door and stepped into the garden.",
    "fox the lazy over jumps brown quick dog the.",
    "Zygote epistemic quasar vortex nebula.",
]

results = [(sentence, perplexity(sentence)) for sentence in TEST_SENTENCES]
results.sort(key=lambda item: item[1])

for sentence, ppl in results:
    print(f"PPL = {ppl:.2f} | {sentence}")
# ===== END MY SOLUTION =====

PPL = 16.66 | She opened the door and stepped into the garden.
PPL = 162.47 | The quick brown fox jumps over the lazy dog.
PPL = 750.36 | Zygote epistemic quasar vortex nebula.
PPL = 8505.68 | fox the lazy over jumps brown quick dog the.


### Task 4 — Find the most surprising token

PPL averages the surprise across all tokens, which hides *where* the model got
confused. For debugging, interpretability, and spotting out-of-distribution inputs,
we often want to know **which single token** had the highest loss.

The helper below returns the per-token loss vector instead of the mean. Print each
token side-by-side with its loss, and flag the worst one.

<div dir="rtl" align="right">

### المهمّة ٤ — ابحث عن الرمز الأكثر مفاجأةً

PPL يُوسّط المفاجأة على كل الرموز، فيُخفي *أين* ارتبك النموذج. نريد غالبًا معرفة
**أيّ رمز بعينه** كانت خسارته الأعلى.

المساعد أدناه يُرجع متجه الخسارة لكل رمز. اطبع كل رمز بجانب خسارته، وأبرِز الأسوأ.

</div>


In [10]:
# === Helper: per-token losses, not just the mean ====================================
import torch.nn.functional as F

@torch.no_grad()
def per_token_losses(text, model, tok):
    """Return (tokens, losses) — one loss per predicted token."""
    enc = tok(text, return_tensors="pt").to(model.device)
    logits = model(**enc).logits                             # (1, T, V)
    # Shift so label at position t is the token the model should predict after seeing 0..t-1
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = enc["input_ids"][:, 1:].contiguous()
    losses = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction="none",
    )
    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0].tolist())
    # losses has length T-1 (we don't predict the first token)
    return tokens[1:], losses.cpu().tolist()

In [11]:
# ────────────────────────────────────────────────────────────────────
# 1) Call per_token_losses(SENTENCE, model, tokenizer) -> (tokens, losses).
# 2) Zip them, print each row as 'token: loss'.
# 3) Use max(enumerate(losses), key=lambda x: x[1]) to find the worst index.
# 4) The worst token is usually the rare name or the semantically broken one.
#
# ١) نادِ per_token_losses(SENTENCE, model, tokenizer) ← (رموز، خسائر).
# ٢) اربطهما وطباعة كل صفّ 'رمز: خسارة'.
# ٣) استخدم max(enumerate(losses), key=lambda x: x[1]) لإيجاد موضع الأسوأ.
# ٤) الأسوأ عادةً هو الاسم النادر أو الجزء المكسور دلاليًّا.
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
SENTENCE = "The capital of Saudi Arabia is Jupiter."

tokens, losses = per_token_losses(SENTENCE, model, tokenizer)

for token, loss in zip(tokens, losses):
    print(f"{token}: {loss:.3f}")

worst_index, worst_loss = max(enumerate(losses), key=lambda x: x[1])
print(f"\nMost surprising token: {tokens[worst_index]}")
print(f"Loss: {worst_loss:.3f}")
# ===== END MY SOLUTION =====

Ġcapital: 9.070
Ġof: 1.807
ĠSaudi: 5.076
ĠArabia: 0.005
Ġis: 1.580
ĠJupiter: 14.094
.: 1.414

Most surprising token: ĠJupiter
Loss: 14.094


### Task 5 — GPT-2 vs aragpt2 on Arabic

Task 1 showed Arabic costs more **tokens**. But that is only half the story. On each
of those tokens, an English-trained model is also **more confused** — so the
per-token loss is higher too. The total cost (number of tokens × per-token
confusion) stacks up fast.

Measure both effects at once by scoring our 3 Arabic sentences with **GPT-2** (which
you already loaded) and then with **aragpt2-base**, and compare the perplexities.

<div dir="rtl" align="right">

### المهمّة ٥ — GPT-2 مقابل aragpt2 على العربيّة

المهمّة ١ أظهرت أن العربيّة تكلّف رموزًا أكثر. وذلك نصف القصّة. على كلٍّ من تلك الرموز،
النموذج الإنجليزيّ **أكثر ارتباكًا** — فخسارة كل رمز أعلى. التكلفة الكلّية (عدد
الرموز × ارتباك كل رمز) تتراكم سريعًا.

قس الأثرين معًا: احسب PPL للجمل العربيّة الثلاث مع **GPT-2** ثم مع **aragpt2-base**،
وقارن.

</div>


In [12]:
# === Load aragpt2-base (we already have the tokenizer; add the model) ==============
ar_model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base").to(DEVICE)
ar_model.eval()
print(f"Loaded aragpt2-base: {sum(p.numel() for p in ar_model.parameters())/1e6:.1f}M params")

model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded aragpt2-base: 135.0M params


In [13]:
# ────────────────────────────────────────────────────────────────────
# 1) perplexity(text) already accepts model and tok as keyword args.
# 2) For GPT-2 use (model, tokenizer); for aragpt2 use (ar_model, ar_tokenizer).
# 3) Print a 2-column comparison. Expect GPT-2 PPL >> aragpt2 PPL on Arabic.
# 4) GPT-2 PPL on Arabic is often in the thousands or tens of thousands.
#
# ١) perplexity(text) يقبل model و tok كوسائط مُسمّاة.
# ٢) لـGPT-2 استخدم (model, tokenizer)، ولـaragpt2 استخدم (ar_model, ar_tokenizer).
# ٣) اطبع مقارنة بعمودين. توقّع PPL(GPT-2) >> PPL(aragpt2) على العربيّة.
# ٤) PPL لـGPT-2 على العربيّة قد يكون بالآلاف أو عشرات الآلاف.
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
from transformers import AutoTokenizer, AutoModelForCausalLM

ar_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base")
ar_model = AutoModelForCausalLM.from_pretrained(
    "aubmindlab/aragpt2-base"
).to(DEVICE)
ar_model.eval()

print("AraGPT2 loaded successfully!")
# ===== END MY SOLUTION =====

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AraGPT2 loaded successfully!


### Takeaway

Three findings, one table:

| | GPT-2 (English) | aragpt2 (Arabic) |
|---|---|---|
| Tokens per Arabic sentence | ~3-4x higher | baseline |
| Perplexity per token | ~100x higher | baseline |
| **Combined cost (API bill, context window)** | **~hundreds of times** | **baseline** |

PPL is **a sanity check**, not a quality metric. A model can have excellent PPL and
still hallucinate; a model can have mediocre PPL and still be useful for your task.
We'll come back to this in **D4** when we fine-tune.

<div dir="rtl" align="right">

### الخلاصة

ثلاث نتائج في جدول واحد:

| | GPT-2 (إنجليزيّ) | aragpt2 (عربيّ) |
|---|---|---|
| رموز لكل جملة عربيّة | أعلى ~٣-٤× | الأساس |
| حيرة لكل رمز | أعلى ~١٠٠× | الأساس |
| **التكلفة المركّبة** | **مئات الأضعاف** | **الأساس** |

PPL **فحص سلامة**، لا مقياس جودة. سنعود لها في **D4** عند الضبط.

</div>


## Section 3 — Controlling the model  *(~35 min)*

Section 2 was about **reading** the model. This section is about **writing with**
it. A language model gives you a probability distribution over the next token;
**decoding** is the policy that turns that distribution into a chosen token, feeds
it back, and repeats.

The strategy you pick controls the trade-off between **safe & repetitive** and
**creative & chaotic**:

| Strategy | What it does | Risk |
|---|---|---|
| **Greedy** | Always pick the argmax | Repetition, loops |
| **Temperature** | Sharpen (T<1) or flatten (T>1) before sampling | T too high → gibberish |
| **Top-k** | Sample only from the k most likely tokens | k too small → repetition |
| **Top-p (nucleus)** | Sample from smallest set with cumulative prob ≥ p | p too high → unsafe |

Four tasks: (1) greedy, (2) temperature sweep, (3) top-k vs top-p, (4) a final
challenge where you tune decoding for two very different prompts.

<div dir="rtl" align="right">

## القسم ٣ — التحكّم في النموذج  *(≈٣٥ دقيقة)*

القسم ٢ كان عن **قراءة** النموذج. هذا القسم عن **الكتابة به**. النموذج يعطيك توزيعًا
احتماليًّا على الرمز التالي؛ **فكّ الترميز** هو السياسة التي تحوّله إلى رمز ثم تعيد
تمريره مدخلًا وتكرّر.

| الاستراتيجيّة | ما تفعله | المخاطرة |
|---|---|---|
| **Greedy** | argmax دومًا | تكرار |
| **Temperature** | تضييق/تفلطح التوزيع قبل العيّنة | هراء |
| **Top-k** | أعلى k رموز فقط | تكرار |
| **Top-p** | أصغر مجموعة احتمالها ≥ p | غير آمن |

أربع مهام: (١) Greedy، (٢) تجربة Temperature، (٣) Top-k مقابل Top-p، (٤) تحدٍّ
ختاميّ تضبط فيه فكّ الترميز لمُحثَّين مختلفين.

</div>


In [14]:
# === Shared prompt for Tasks 1-3 ====================================================
PROMPT = "The best thing about artificial intelligence is"
print("PROMPT:", PROMPT)

PROMPT: The best thing about artificial intelligence is


### Task 1 — Greedy decoding

Set `do_sample=False` to always pick the argmax token. Run it twice: outputs should
be **identical** (greedy is deterministic). Watch for repetition toward the end on
this small model.

<div dir="rtl" align="right">

### المهمّة ١ — فكّ ترميز Greedy

اضبط `do_sample=False` لأخذ argmax دومًا. شغّلها مرّتين: المخرجات يجب أن تكون
**متطابقة** (Greedy حتميّة). راقب التكرار في النهاية.

</div>


In [15]:
# ────────────────────────────────────────────────────────────────────
# 1) Greedy = argmax every step. In HuggingFace, pass do_sample=False.
# 2) Expect short-term fluency, then repetition on this small model.
# 3) Greedy is deterministic — running twice gives identical output.
#
# ١) Greedy = argmax كل خطوة. في HuggingFace مرّر do_sample=False.
# ٢) توقّع طلاقة قصيرة ثم تكرارًا على هذا النموذج الصغير.
# ٣) Greedy حتميّة — التشغيل مرّتين يعطي نفس المخرج.
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
output1 = generate(PROMPT, max_new_tokens=30, do_sample=False)
output2 = generate(PROMPT, max_new_tokens=30, do_sample=False)

print("First output:")
print(output1)

print("\nSecond output:")
print(output2)

print("\nIdentical:", output1 == output2)
# ===== END MY SOLUTION =====

First output:
that it's not just a tool for the job. It's a tool for the job.

"It's a tool for the job.

Second output:
that it's not just a tool for the job. It's a tool for the job.

"It's a tool for the job.

Identical: True


### Task 2 — Temperature sweep

Enable sampling (`do_sample=True`) and sweep `temperature` across 0.3, 0.8, 1.5. Set
the seed before each call so you can compare. Watch for fluency breaking down at
high T.

<div dir="rtl" align="right">

### المهمّة ٢ — تجربة Temperature

فعّل العيّنة (`do_sample=True`) وجرّب `temperature` بقيم ٠٫٣، ٠٫٨، ١٫٥. ثبّت البذرة
قبل كل نداء للمقارنة. راقب انهيار الطلاقة عند T مرتفعة.

</div>


In [16]:
# ────────────────────────────────────────────────────────────────────
# 1) Pass do_sample=True with temperature values 0.3, 0.8, 1.5.
# 2) torch.manual_seed before each call makes results reproducible.
# 3) Expect: 0.3 cautious, 0.8 natural, 1.5 increasingly unhinged.
#
# ١) مرّر do_sample=True مع temperature بقيم ٠٫٣، ٠٫٨، ١٫٥.
# ٢) torch.manual_seed قبل كل نداء يجعل النتائج قابلة للإعادة.
# ٣) توقّع: ٠٫٣ متحفّظ، ٠٫٨ طبيعيّ، ١٫٥ يزداد جموحًا.
# ────────────────────────────────────────────────────────────────────
# ===== BEGIN MY SOLUTION =====
for temperature in [0.3, 0.8, 1.5]:
    torch.manual_seed(42)
    output = generate(
        PROMPT,
        max_new_tokens=30,
        do_sample=True,
        temperature=temperature
    )
    print(f"\nTemperature = {temperature}")
    print(output)
# ===== END MY SOLUTION =====


Temperature = 0.3
that it's so easy to create, and so easy to use, and so easy to use.

What's the best way to create and

Temperature = 0.8
that you can't learn something by watching people. So you have to learn it from experience."

To help people learn, the researchers also turned

Temperature = 1.5
for you take that advantage as much, and not have it cost too big." For IBM, "coupled with all sorts of big business innovations


### Task 3 — Top-k vs Top-p

Both techniques crop the long tail of unlikely tokens so sampling stays safe. Use
`temperature=0.7` for both — the production standard — and compare which output
feels more natural.

<div dir="rtl" align="right">

### المهمّة ٣ — Top-k مقابل Top-p

كلاهما يقصّ الذيل الطويل من الرموز المستبعدة لإبقاء العيّنة آمنة. استخدم
`temperature=0.7` لكليهما — المعيار الإنتاجيّ — وقارن أيّهما أطبع.

</div>


In [17]:
# ────────────────────────────────────────────────────────────────────
# 1) top_k=50 restricts sampling to the 50 most likely tokens at each step.
# 2) top_p=0.9 keeps the smallest set whose cumulative prob >= 0.9.
# 3) Use temperature=0.7 with both — the production standard.
#
# ١) top_k=50 يقصر العيّنة على أعلى ٥٠ رمزًا.
# ٢) top_p=0.9 يبقي أصغر مجموعة احتمالها التراكميّ ≥ ٠٫٩.
# ٣) استخدم temperature=0.7 مع كليهما — المعيار الإنتاجيّ.
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
torch.manual_seed(42)
top_k_output = generate(
    PROMPT,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_k=50
)

torch.manual_seed(42)
top_p_output = generate(
    PROMPT,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print("Top-k output:")
print(top_k_output)

print("\nTop-p output:")
print(top_p_output)
# ===== END MY SOLUTION =====

Top-k output:
that you can't learn something by watching it."

The latest version of Microsoft's artificial intelligence tool, called Cortana, has been released. The

Top-p output:
that you can't learn something by watching it."

The latest version of Microsoft's artificial intelligence tool, called Cortana, has been released. The


### Task 4 — Tune decoding for two different prompts

You've seen each knob in isolation. Real work is almost never one prompt — it's a
pipeline of different prompts, each wanting different settings.

For each prompt below, find decoding settings that produce the **best** output *for
that prompt's purpose*. There is no "right answer" — only reasoned choices. Expect:

- **Factual prompt** → the answer is a specific fact. You want **low temperature,
  maybe greedy**. Variety is bad here: it's just a chance to be wrong.
- **Creative prompt** → the point *is* variety. You want **higher temperature +
  top-p**. Greedy here produces boring, repetitive text.

Try at least **two different settings per prompt** and pick a winner. In the final
cell, write one short English sentence explaining why your choices differed.

<div dir="rtl" align="right">

### المهمّة ٤ — اضبط فكّ الترميز لمُحثَّين مختلفين

رأيتَ كل مقبض على حدة. العمل الحقيقيّ ليس مُحثًّا واحدًا، بل خطّ أنابيب من مُحثَّات
مختلفة، كلٌّ يريد إعدادًا مختلفًا.

لكلّ مُحثّ أدناه، اعثر على إعدادات فكّ ترميز تُنتج **أفضل** مخرج *لهدفه*. لا إجابة
"صحيحة" — فقط اختيارات مُبرَّرة.

- **المُحثّ المعلوماتيّ** ← الإجابة حقيقة محدّدة. تريد **temperature منخفض أو Greedy**.
- **المُحثّ الإبداعيّ** ← الهدف *هو* التنوّع. تريد **temperature مرتفع + top-p**.

جرّب **إعدادين على الأقل لكل مُحثّ** واختر الفائز. في الخلية الأخيرة اكتب جملة
إنجليزيّة قصيرة تشرح لماذا اختلفت اختياراتك.

</div>


In [18]:
# ────────────────────────────────────────────────────────────────────
# 1) FACTUAL: try do_sample=False (greedy) OR a very low temperature like 0.2.
# 2) CREATIVE: try temperature=0.9 with top_p=0.9, and maybe a longer max_new_tokens.
# 3) Call generate(...) twice per prompt with different settings, print both.
# 4) The "best" decoding is NOT the same across tasks — that is the whole point.
#
# ١) معلوماتيّ: جرّب do_sample=False (Greedy) أو temperature منخفض جدًّا كـ٠٫٢.
# ٢) إبداعيّ: جرّب temperature=0.9 مع top_p=0.9، وربّما max_new_tokens أطول.
# ٣) نادِ generate(...) مرّتين لكل مُحثّ بإعدادات مختلفة، واطبع كليهما.
# ٤) "أفضل" فكّ ترميز ليس نفسه بين المهام — وهذا هو الدرس.
# ────────────────────────────────────────────────────────────────────

# ===== BEGIN MY SOLUTION =====
factual_prompt = "The capital of Saudi Arabia is"

print("Factual - Greedy:")
print(generate(factual_prompt, max_new_tokens=20, do_sample=False))

print("\nFactual - Low temperature:")
torch.manual_seed(42)
print(generate(
    factual_prompt,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.2
))

creative_prompt = "Write a creative opening for a story about a robot"

print("\nCreative - Temperature 0.7:")
torch.manual_seed(42)
print(generate(
    creative_prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
))

print("\nCreative - Temperature 1.2:")
torch.manual_seed(42)
print(generate(
    creative_prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=1.2,
    top_p=0.9
))
# ===== END MY SOLUTION =====

Factual - Greedy:
the capital of the kingdom's oil industry, and its oil production is estimated to be worth $1

Factual - Low temperature:
home to the world's largest oil reserves, and the country's economy is growing at a rapid clip

Creative - Temperature 0.7:
, you can't do it without a story. So, in this case, it's going to be a robot story. You're going to have to have a story about a robot that you're

Creative - Temperature 1.2:
, you can't expect something better than this. So, in closing, it's not all roses, but it's all fun and excitement.


### Write-up

In **one English sentence**, explain *why* the right decoding for the factual prompt
is different from the right decoding for the creative prompt. (Hint: think about
what counts as "correct" in each case.)

> *Example answer: "For a factual prompt there is exactly one correct next token,
> so sampling adds nothing but risk; for a creative prompt there are many good
> continuations, so sampling is where the quality comes from."*

<div dir="rtl" align="right">

### كتابة

في **جملة إنجليزيّة واحدة**، اشرح *لماذا* يختلف فكّ الترميز الصحيح للمُحثّ المعلوماتيّ
عن الإبداعيّ. (تلميح: ما الذي يُعدّ "صحيحًا" في كلّ حالة؟)
Factual prompts require accuracy and consistency, while creative prompts prioritize variety and originality, so they need different decoding settings.
</div>


## Wrap-up

Four things you can now do that you couldn't an hour ago:

1. **Open any LLM API bill** and know, within a factor of 2, whether a given
   sentence will be charged more or less because of the language.
2. **Debug a sentence** that confuses the model by finding the single most
   surprising token in it.
3. **Read a model card's "PPL = X" line** and translate it to a mental image of the
   model's surprise level on that benchmark.
4. **Tune decoding** for the task at hand: greedy for facts, nucleus sampling for
   open-ended generation.

Those four skills are the whole mental model you need to walk into **D2** (modern
LLMs + RAG), where we stop looking at a single model and start combining a
retriever with a generator to answer questions grounded in documents.

<div dir="rtl" align="right">

## الخلاصة

أربعة أمور تستطيع الآن:

١. **تفتح فاتورة أيّ LLM API** وتعرف، تقريبًا، هل ستُحاسَب على جملة أكثر أو أقلّ.
٢. **تُصحّح جملة** تربك النموذج بإيجاد الرمز الأكثر مفاجأةً فيها.
٣. **تقرأ سطر "PPL = X"** في بطاقة النموذج وتحوّله لصورة ذهنيّة.
٤. **تضبط فكّ الترميز** لمهمّتك: Greedy للحقائق، nucleus للتوليد المفتوح.

وهذه الصورة الذهنيّة التي تحتاجها لدخول **D2** (النماذج الحديثة + RAG).

</div>
